# 02_modelado — Modelo baseline de riesgo de stunting (16 features basales)

**Entrada unica:** `data/processed/model_dataset.csv` (generado por el Paso 9.2 del EDA).

**Contrato:** 16 features basales (10 maternas + 6 del nacimiento) -> targets `stunted_12` y `stunted_24`. Split 70/15/15 estratificado por bebe; imputacion/escalado SOLO en train; metricas F1, PR-AUC, ROC-AUC y sensibilidad; `class_weight='balanced'`.

**MLOps:** seed fija, hash MD5 del dataset en metadatos y MLflow, modelos en `models/`, figuras en `figures/`.

## 1. Imports y configuracion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, f1_score, recall_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve)

import joblib
import hashlib
import json
from pathlib import Path

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False
    print('[INFO] MLflow no instalado; se omitira el logging.')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

SEED = 42
np.random.seed(SEED)

import sklearn
print('pandas', pd.__version__)
print('numpy', np.__version__)
print('sklearn', sklearn.__version__)
if MLFLOW_AVAILABLE:
    print('mlflow', mlflow.__version__)

## 2. Rutas y configuracion

In [ ]:
def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / '.git').exists() or (p / 'README.md').exists():
            return p
    return start

ROOT = find_root()
DATA_PATH = ROOT / 'data' / 'processed' / 'model_dataset.csv'
FIG_DIR = ROOT / 'figures' / '02_model'        # <- subcarpeta exclusiva de este notebook
MODELS_DIR = ROOT / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_ENROL = ['enrol_hiv_status_cat', 'momage_cat', 'educ_cat_n', 'marital_cat',
                  'hfia_enr', 'wealth_quintile', 'depression', 'mom_muac_cat',
                  'parity', 'enrol_anemia']
FEATURES_NAC = ['b1_sex', 'gestage_final', 'caesarean', 'preterm', 'sga', 'lbw']
FEATURES = FEATURES_ENROL + FEATURES_NAC
TARGETS = ['stunted_12', 'stunted_24']
ID_COL = 'newid'

CONFIG = {'data_path': str(DATA_PATH), 'figures_dir': str(FIG_DIR),
          'models_dir': str(MODELS_DIR), 'n_features': len(FEATURES),
          'targets': TARGETS, 'seed': SEED, 'class_weight': 'balanced'}

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))      # para importar src/ desde notebooks/

print('ROOT:', ROOT)
print('DATA:', DATA_PATH)
print('FIGS:', FIG_DIR)
print('MODELS:', MODELS_DIR)

## 3. Carga y validacion del dataset plano

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError('No existe ' + str(DATA_PATH) + '. Corre primero el Paso 9.2 del EDA para generar data/processed/model_dataset.csv.')

dataset = pd.read_csv(DATA_PATH)
print('Shape:', dataset.shape)
print('Bebes unicos:', dataset[ID_COL].nunique())
dataset.head()

In [ ]:
required = [ID_COL] + FEATURES + TARGETS
missing_cols = [c for c in required if c not in dataset.columns]
if missing_cols:
    raise ValueError('Faltan columnas en model_dataset.csv: ' + str(missing_cols))
print('Esquema OK:', len(required), 'columnas requeridas presentes.')

miss = dataset[FEATURES].isna().sum()
print('Missing por feature:')
print(miss[miss > 0])

for t in TARGETS:
    print()
    print(t)
    print(dataset[t].value_counts(dropna=False))

## 4. Definicion de X/y y split 70/15/15 (sin traslape de bebes)

In [ ]:
# ============================================================================
# Definicion de X/y y split 70/15/15 (sin traslape de bebes)
# ============================================================================
X_all = dataset[FEATURES].copy()

def make_splits(target_col, seed=SEED):
    """Construye y = target mapeado, descarta bebés sin target en ese horizonte
    y particiona 70/15/15 estratificado (una fila por bebé => sin fuga por grupo)."""
    y = dataset[target_col].map({'no': 0, 'yes': 1})
    mask = y.notna()                                  # <- bebés SIN visita en ese horizonte fuera
    X = X_all[mask]
    y = y[mask].astype(int)
    g = dataset[ID_COL][mask]
    print(f'{target_col}: {len(X)} bebés válidos ({(~mask).sum()} sin target descartados)')

    X_tr, X_tmp, y_tr, y_tmp, g_tr, g_tmp = train_test_split(
        X, y, g, test_size=0.30, stratify=y, random_state=seed)
    X_va, X_te, y_va, y_te, g_va, g_te = train_test_split(
        X_tmp, y_tmp, g_tmp, test_size=0.50, stratify=y_tmp, random_state=seed)

    assert set(g_tr).isdisjoint(g_va)
    assert set(g_tr).isdisjoint(g_te)
    assert set(g_va).isdisjoint(g_te)
    return X_tr, X_va, X_te, y_tr, y_va, y_te, g_tr, g_va, g_te

splits_12 = make_splits('stunted_12')
splits_24 = make_splits('stunted_24')

for name, s in [('stunted_12', splits_12), ('stunted_24', splits_24)]:
    print(name, '| train', len(s[0]), round(s[3].mean() * 100, 1), '% + | val', len(s[1]),
          round(s[4].mean() * 100, 1), '% + | test', len(s[2]), round(s[5].mean() * 100, 1), '% +')

## 5. Pipeline de preprocesamiento (anti-leakage: se ajusta solo en train)

El preprocesador se importa de `src/preprocessing.py`, la misma fuente que usan `train_stunting.py`, `evaluate_cv.py`, `predict.py` y el tablero. Definir `Winsorizer` dentro del notebook hacia que el pickle la referenciara como `__main__.Winsorizer` y el modelo no pudiera cargarse desde ningun otro proceso.

In [ ]:
from src.preprocessing import (Winsorizer, build_preprocessor,
                               NUMERICAS, CAT_SIN_MISSING, CAT_CON_MISSING)

NUM_FEATURES, CAT_OK, CAT_NA = NUMERICAS, CAT_SIN_MISSING, CAT_CON_MISSING
assert set(NUM_FEATURES + CAT_OK + CAT_NA) == set(FEATURES), 'el contrato del notebook y el de src/ difieren'

# Bloques: num (winsor 25-44 + mediana + escala), cat_ok (moda + one-hot),
#          cat_na (categoria explicita "missing" + one-hot)
preprocessor = build_preprocessor()
print('Preprocesador listo:', [nombre for nombre, _, _ in preprocessor.transformers])
print('Winsorizer viene de', Winsorizer.__module__)

## 6. Modelos baseline

In [ ]:
# ============================================================================
# Modelos baseline (configs alineadas con el sweep MLflow, 6 runs)
#  - RF 200/6/4 = run RF_base: mejor compromiso test (AUC .506, PR-AUC .392);
#    profundidad 6 evita el sobreajuste de profundidad libre (RF_complex, gap .28)
#  - LR C=1.0   = mejor PR-AUC test que C=0.1 (.330 vs .274)
#  - GB         = referencia conservadora (no soporta class_weight; ver run GB)
# ============================================================================
models = {
    'logistic_regression': LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=SEED),
    'random_forest': RandomForestClassifier(n_estimators=200, max_depth=6, max_features=4,
                                            class_weight='balanced', random_state=SEED, n_jobs=-1),
    'gradient_boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                                    max_depth=3, random_state=SEED),
}
print('Modelos baseline:', list(models.keys()))

## 7. Entrenamiento y evaluacion

In [ ]:
def evaluate_predictions(y_true, y_pred, y_proba):
    return {'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'sensitivity': recall_score(y_true, y_pred, zero_division=0),
            'pr_auc': average_precision_score(y_true, y_proba),
            'roc_auc': roc_auc_score(y_true, y_proba)}

def train_eval_model(model, X_tr, X_va, X_te, y_tr, y_va, y_te):
    # clone(): cada llamada entrena objetos nuevos. Sin esto, 12m y 24m compartian el
    # mismo estimador y el fit de 24m sobreescribia al de 12m (ambos .joblib salian iguales).
    pipe = Pipeline([('preprocessor', clone(preprocessor)), ('classifier', clone(model))])
    pipe.fit(X_tr, y_tr)
    out = {}
    for tag, Xs, ys in [('train', X_tr, y_tr), ('val', X_va, y_va), ('test', X_te, y_te)]:
        out[tag] = evaluate_predictions(ys, pipe.predict(Xs), pipe.predict_proba(Xs)[:, 1])
    return pipe, out

print('Funciones de evaluacion listas.')

In [ ]:
print('=' * 60)
print('HORIZONTE stunted_12')
print('=' * 60)
X_tr, X_va, X_te, y_tr, y_va, y_te, *_ = splits_12
results_12 = {}
for name, model in models.items():
    pipe, metrics = train_eval_model(model, X_tr, X_va, X_te, y_tr, y_va, y_te)
    results_12[name] = {'pipeline': pipe, 'metrics': metrics}
    print()
    print(name)
    print('  val :', {k: round(v, 3) for k, v in metrics['val'].items()})
    print('  test:', {k: round(v, 3) for k, v in metrics['test'].items()})

In [ ]:
print('=' * 60)
print('HORIZONTE stunted_24')
print('=' * 60)
X_tr, X_va, X_te, y_tr, y_va, y_te, *_ = splits_24
results_24 = {}
for name, model in models.items():
    pipe, metrics = train_eval_model(model, X_tr, X_va, X_te, y_tr, y_va, y_te)
    results_24[name] = {'pipeline': pipe, 'metrics': metrics}
    print()
    print(name)
    print('  val :', {k: round(v, 3) for k, v in metrics['val'].items()})
    print('  test:', {k: round(v, 3) for k, v in metrics['test'].items()})

## 8. Comparacion de resultados

In [ ]:
rows = []
for target, results in [('stunted_12', results_12), ('stunted_24', results_24)]:
    for model_name, info in results.items():
        for split, mets in info['metrics'].items():
            row = {'target': target, 'model': model_name, 'split': split}
            row.update(mets)
            rows.append(row)
results_df = pd.DataFrame(rows)
print('VALIDACION:')
display(results_df[results_df['split'] == 'val'].sort_values(['target', 'f1'], ascending=[True, False]))
print('TEST:')
display(results_df[results_df['split'] == 'test'].sort_values(['target', 'f1'], ascending=[True, False]))

In [ ]:
def best_by_val_f1(results):
    return max(results.items(), key=lambda kv: kv[1]['metrics']['val']['f1'])

best_model_12, best_info_12 = best_by_val_f1(results_12)
best_model_24, best_info_24 = best_by_val_f1(results_24)
best_pipeline_12 = best_info_12['pipeline']
best_pipeline_24 = best_info_24['pipeline']
print('Mejor 12m:', best_model_12, '| F1 val:', round(best_info_12['metrics']['val']['f1'], 4))
print('Mejor 24m:', best_model_24, '| F1 val:', round(best_info_24['metrics']['val']['f1'], 4))

## 9. Figuras para el reporte (se guardan en figures/ de la raiz)

In [ ]:
X_te12, y_te12 = splits_12[2], splits_12[5]
X_te24, y_te24 = splits_24[2], splits_24[5]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_estimator(best_pipeline_12, X_te12, y_te12, ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de confusion test - stunted_12 (' + best_model_12 + ')')
ConfusionMatrixDisplay.from_estimator(best_pipeline_24, X_te24, y_te24, ax=axes[1], cmap='Blues')
axes[1].set_title('Matriz de confusion test - stunted_24 (' + best_model_24 + ')')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

proba_12 = best_pipeline_12.predict_proba(X_te12)[:, 1]
proba_24 = best_pipeline_24.predict_proba(X_te24)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fpr1, tpr1, _ = roc_curve(y_te12, proba_12)
fpr2, tpr2, _ = roc_curve(y_te24, proba_24)
axes[0].plot(fpr1, tpr1, label='stunted_12')
axes[0].plot(fpr2, tpr2, label='stunted_24')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_title('Curvas ROC (test)')
axes[0].legend()
prec1, rec1, _ = precision_recall_curve(y_te12, proba_12)
prec2, rec2, _ = precision_recall_curve(y_te24, proba_24)
axes[1].plot(rec1, prec1, label='stunted_12')
axes[1].plot(rec2, prec2, label='stunted_24')
axes[1].set_title('Curvas Precision-Recall (test)')
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_curvas_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()

clf = best_pipeline_24.named_steps['classifier']
if hasattr(clf, 'feature_importances_'):
    names = best_pipeline_24.named_steps['preprocessor'].get_feature_names_out()
    imp = clf.feature_importances_
    def orig_var(n):
        pref, rest = n.split('__')
        return rest if pref == 'num' else rest.rsplit('_', 1)[0]
    serie = pd.Series(imp, index=[orig_var(n) for n in names]).groupby(level=0).sum().sort_values(ascending=False)
    top = serie.head(10)
    plt.figure(figsize=(9, 5))
    plt.barh(top.index[::-1], top.values[::-1], color='#e67e22')
    plt.title('Top 10 variables por importancia - mejor modelo 24m')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig7_importancia_variables.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('El mejor modelo 24m no expone feature_importances_; se omite esa figura.')

## 10. Guardado de modelos y metadatos (models/)

In [ ]:
def file_md5(path):
    """MD5 del contenido con saltos de linea normalizados: el mismo valor en Windows
    (CRLF por autocrlf) y en Linux (LF), y el mismo que registran evaluate_cv y la escalera."""
    return hashlib.md5(Path(path).read_bytes().replace(b'\r\n', b'\n')).hexdigest()

data_hash = file_md5(DATA_PATH)
print('MD5 del dataset:', data_hash)

model_12_path = MODELS_DIR / ('model_stunting_12m_' + best_model_12 + '.joblib')
model_24_path = MODELS_DIR / ('model_stunting_24m_' + best_model_24 + '.joblib')
joblib.dump(best_pipeline_12, model_12_path)
joblib.dump(best_pipeline_24, model_24_path)

## Metadata actualizada con la decision final tras el sweep MLflow + re-evaluacion local:
## LR empaquetado en ambos horizontes (criterio val F1, consistente VM y local);
## RF 200/6/4 del sweep queda como referencia arborea validada.
metadata = {'data_source': DATA_PATH.relative_to(ROOT).as_posix(), 'data_hash': data_hash, 'features': FEATURES,
            'targets': TARGETS, 'seed': SEED, 'class_weight': 'balanced',
            'best_model_12': best_model_12, 'best_model_24': best_model_24,
            'metrics_12': best_info_12['metrics'], 'metrics_24': best_info_24['metrics'],
            'mlflow_sweep': 'stunting-baseline-multi (6 runs: RF x3, LR x2, GB x1) - EC2 13.217.27.253:8050',
            'criterio_seleccion': 'val F1 (best_by_val_f1); LR ganador en VM (0.513) y en local (0.24 / 0.345)',
            'decision_modelos': 'LR empaquetado en ambos horizontes; RF 200/6/4 = referencia arborea validada (empate RF≈LR, senal basal debil)'}
with open(MODELS_DIR / 'model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Guardado:', model_12_path.name)
print('Guardado:', model_24_path.name)
print('Guardado: model_metadata.json')

# Verificacion: los dos modelos son distintos y cargan desde un proceso limpio
import subprocess
assert hashlib.md5(model_12_path.read_bytes()).hexdigest() != hashlib.md5(model_24_path.read_bytes()).hexdigest(), \
    'los modelos de 12m y 24m son identicos: revisar clone() en train_eval_model'
chk = subprocess.run([sys.executable, '-c',
                      'import joblib,sys; sys.path.insert(0, sys.argv[1]); joblib.load(sys.argv[2]); joblib.load(sys.argv[3]); print("OK")',
                      str(ROOT), str(model_12_path), str(model_24_path)], capture_output=True, text=True)
print('Carga desde otro proceso:', chk.stdout.strip() or chk.stderr.strip()[-200:])
assert chk.returncode == 0, 'los modelos no cargan fuera del notebook'

## 11. Logging opcional en MLflow

In [ ]:
# Opcional: para loguear los ganadores en el server de la VM, pon aqui la URI
# y reinicia el server con --default-artifact-root mlflow-artifacts:/
MLFLOW_TRACKING_URI = ''   # ej.: 'http://13.217.27.253:8050'

if MLFLOW_AVAILABLE:
    if MLFLOW_TRACKING_URI:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment('stunting_baseline_model_dataset')
    runs = [('stunted_12', best_model_12, best_info_12, best_pipeline_12, splits_12[0]),
            ('stunted_24', best_model_24, best_info_24, best_pipeline_24, splits_24[0])]
    for target, best_name, best_info, pipe, X_tr in runs:
        with mlflow.start_run(run_name='baseline_' + target + '_' + best_name):
            mlflow.log_param('target', target)
            mlflow.log_param('data_hash', data_hash)
            mlflow.log_param('features_type', '16_basales')
            mlflow.log_param('best_model', best_name)
            mlflow.log_param('seed', SEED)
            for split, mets in best_info['metrics'].items():
                for k, v in mets.items():
                    mlflow.log_metric(split + '_' + k, v)
            try:
                mlflow.sklearn.log_model(pipe, name='model', input_example=X_tr.head(5))
            except TypeError:
                mlflow.sklearn.log_model(pipe, artifact_path='model', input_example=X_tr.head(5))
    print('MLflow logging completado.')
else:
    print('[INFO] Sin MLflow; logging omitido.')

## 12. Resumen y proximos pasos

- Baseline tiempo-cero (16 features basales) entrenado, evaluado y versionado para ambos horizontes.
- Configs alineadas con el sweep MLflow (6 runs): RF 200/6/4 = compromiso validado en 24m; LR C=1.0 en 12m; profundidad libre descartada por sobreajuste (gap 0.28).
- Hallazgos del sweep: techo de senal basal (test AUC 0.44-0.51 en 3 familias); features robustas wealth_quintile / gestage_final / hfia_enr (+ sga en lineales).
- Proximo experimento: agregar features postnales tempranas (Z-scores week-3/month-3 y pendiente dLAZ) y comparar en MLflow.
- Despues: definicion del punto de operacion (umbral) en el tablero; los modelos ya exponen predict_proba.